# Run Variant Pipeline

| Variant | NLI_REORDER | NLI_HINT |
|---|---|---|
| `a0` | 0 | 0 |
| `a1` | 1 | 0 |
| `a2` | 0 | 1 |
| `v_new` | 1 | 1 |

## Config

In [2]:
VARIANT_ID = 'a1'
REPORT_IDS = [
    'Bangchak2024', 'Bangchak2025', 'Bapco2023', 'Bapco2024', 'Bunduq2023',
    'Desfa2023', 'Energean2023', 'Energean2024', 'HPCL2023', 'HPCL2024',
    'IslandOil2024', 'Mubadala2023', 'Mubadala2024', 'OKQ82023', 'OKQ82024', 'OQ2023',
]
FORCE = False

_VARIANT_FLAGS = {
    'a0':    {'NLI_REORDER_ENABLED': '0', 'NLI_HINT_ENABLED': '0'},
    'a1':    {'NLI_REORDER_ENABLED': '1', 'NLI_HINT_ENABLED': '0'},
    'a2':    {'NLI_REORDER_ENABLED': '0', 'NLI_HINT_ENABLED': '1'},
    'v_new': {'NLI_REORDER_ENABLED': '1', 'NLI_HINT_ENABLED': '1'},
}

_flags = _VARIANT_FLAGS[VARIANT_ID]
print(f'Variant: {VARIANT_ID} | REORDER={_flags["NLI_REORDER_ENABLED"]} | HINT={_flags["NLI_HINT_ENABLED"]} | FORCE={FORCE} | {len(REPORT_IDS)} reports')

Variant: a1 | REORDER=1 | HINT=0 | FORCE=False | 16 reports


## Setup

In [3]:
import os
import sys
import asyncio
import shutil
import time
import traceback
from pathlib import Path
from dotenv import load_dotenv
from datetime import datetime, timezone
import pandas as pd

REPO_ROOT = Path(r'D:\Final_GRAG')
EXP_DIR = REPO_ROOT / 'experiments' / 'module3_nli'
DEST_ROOT = EXP_DIR / 'variant_runs'
sys.path.insert(0, str(REPO_ROOT))
from nli_lib.constants import LATENCY_OBSERVED_CSV, LATENCY_DIR
LATENCY_LOG = LATENCY_OBSERVED_CSV
LATENCY_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(REPO_ROOT / '.env')

os.environ.update(_flags)
print(f'NLI_REORDER_ENABLED={os.environ["NLI_REORDER_ENABLED"]} | NLI_HINT_ENABLED={os.environ["NLI_HINT_ENABLED"]}')

NLI_REORDER_ENABLED=1 | NLI_HINT_ENABLED=0


In [4]:
from src.compliance.graph import build_compliance_graph
from src.compliance import io as io_mod
import src.compliance.config as cfg

graph = build_compliance_graph(checkpoint=False)

## Run variants

In [5]:
def _is_done(rid):
    return (DEST_ROOT / VARIANT_ID / rid / 'compliance_report.json').exists()

done = [r for r in REPORT_IDS if _is_done(r)] if not FORCE else []
pending = [r for r in REPORT_IDS if r not in done]
print(f'{len(done)} already done | {len(pending)} pending')

16 already done | 0 pending


In [6]:
async def _run_one_report(rid):
    t0 = time.time()
    # Khởi tạo state
    state = {
        'report_id': rid,
        'inputs': io_mod.load_report_inputs(rid),
        'phase_results': {},
        'pending_omissions': [],
    }
    final_state = await graph.ainvoke(state)
    
    elapsed = time.time() - t0
    io_mod.write_outputs(final_state)
    final_pr = final_state.get('phase_results', {}).get('final')
    overall = bool(final_pr.artifacts.get('overall_pass', False)) if final_pr else False
    return {'rid': rid, 'overall': 'PASS' if overall else 'FAIL', 'elapsed_s': round(elapsed, 1)}

In [7]:
def _copy_to_variant_runs(rid):
    src_dir = Path(cfg.REPORT_UNITS_DIR) / rid
    dst_dir = DEST_ROOT / VARIANT_ID / rid
    dst_dir.mkdir(parents=True, exist_ok=True)
    for fname in ('compliance_report.json', 'compliance_summary.csv'):
        src = src_dir / fname
        if src.exists():
            shutil.copy2(src, dst_dir / fname)

In [8]:
async def _run_all():
    out = []
    n_total = len(REPORT_IDS)
    for rank, rid in enumerate(REPORT_IDS):
        if rid in done:
            continue
        ts = datetime.now(timezone.utc).strftime('%H:%M:%S')
        print(f'[{ts}] [{rank + 1}/{n_total}] {rid}: running...')
        t0 = time.time()
        try:
            res = await _run_one_report(rid)
            t1 = time.time()
            print(f'    {res["overall"]} | {res["elapsed_s"]}s')
            _copy_to_variant_runs(rid)
            out.append(res)

            # Ghi 1 dòng log cho report vừa chạy xong; replace nếu (variant, report_id) đã tồn tại
            new_row = pd.DataFrame([{
                'variant': VARIANT_ID,
                'rank': rank,
                'report_id': rid,
                'finish_time': datetime.fromtimestamp(t1).isoformat(timespec='seconds'),
                'duration_min': round((t1 - t0) / 60, 2),
            }])
            if LATENCY_LOG.exists():
                df = pd.read_csv(LATENCY_LOG)
                df = df[~((df['variant'] == VARIANT_ID) & (df['report_id'] == rid))]
                df = pd.concat([df, new_row], ignore_index=True)
            else:
                df = new_row
            df.to_csv(LATENCY_LOG, index=False)
        except KeyboardInterrupt:
            print('KeyboardInterrupt; stopping')
            break
        except Exception as e:
            print(f'    ERROR: {e!r}')
            traceback.print_exc()
            out.append({'rid': rid, 'overall': 'ERROR', 'elapsed_s': None})
    return out

In [9]:
if pending:
    results = asyncio.run(_run_all())
    n_pass = sum(1 for r in results if r['overall'] == 'PASS')
    n_fail = sum(1 for r in results if r['overall'] == 'FAIL')
    n_err  = sum(1 for r in results if r['overall'] == 'ERROR')
    print(f'Done: {n_pass} PASS / {n_fail} FAIL / {n_err} ERROR')
else:
    print('Nothing to run.')

Nothing to run.


## Preview log

In [10]:
if LATENCY_LOG.exists():
    _log = pd.read_csv(LATENCY_LOG)
    print(_log[_log['variant'] == VARIANT_ID].to_string(index=False))
else:
    print(f'No log file at {LATENCY_LOG}')

variant  rank     report_id         finish_time  duration_min
     a1     0  Bangchak2024 2026-05-07T09:45:07           NaN
     a1     1  Bangchak2025 2026-05-07T10:29:51         44.74
     a1     2     Bapco2023 2026-05-07T11:01:05         31.23
     a1     3     Bapco2024 2026-05-07T11:33:38         32.54
     a1     4    Bunduq2023 2026-05-07T12:04:49         31.18
     a1     5     Desfa2023 2026-05-07T12:38:50         34.02
     a1     6  Energean2023 2026-05-07T12:39:40           NaN
     a1     7  Energean2024 2026-05-07T13:22:49         43.16
     a1     8      HPCL2023 2026-05-07T14:10:34         47.75
     a1     9      HPCL2024 2026-05-07T14:50:42         40.14
     a1    10 IslandOil2024 2026-05-07T15:14:37         23.91
     a1    11  Mubadala2023 2026-05-07T15:57:11         42.58
     a1    12  Mubadala2024 2026-05-07T16:37:07         39.92
     a1    13      OKQ82023 2026-05-07T17:15:15         38.14
     a1    14      OKQ82024 2026-05-07T17:52:05         36.83
     a1 

## Tổng kết

In [11]:
import json

LLM_PHASES = ['phase3', 'phase5', 'phase6']
n_verdicts = 0
n_disclosures = 0

for rid in REPORT_IDS:
    path = DEST_ROOT / VARIANT_ID / rid / 'compliance_report.json'
    if not path.exists():
        continue
    report = json.loads(path.read_text(encoding='utf-8'))
    for phase_key in LLM_PHASES:
        phase = report['phase_results'][phase_key]
        for disc in phase['artifacts']['disclosure_verdicts']:
            n_disclosures += 1
            n_verdicts += len(disc['requirement_verdicts'])

print(f'disclosures: {n_disclosures:,}')
print(f'verdicts:    {n_verdicts:,}')

disclosures: 1,845
verdicts:    8,660


## Case bất đồng

In [12]:
import json
from nli_lib.constants import VARIANTS, MATERIAL_TOPIC_NA_SENTINEL

_PAIR_PHASES = ['phase3', 'phase5', 'phase6']
_REAL = {'pass', 'partial', 'fail', 'no_evidence'}
_TAG_UNREC = '_unrecoverable_'
_JOIN = ['report_id', 'phase', 'disclosure_id', 'material_topic', 'requirement_id', 'occurrence_idx']
_DISC_KEYS = ['report_id', 'disclosure_id', 'material_topic', 'occurrence_idx']

def _load_report(variant, rid):
    path = DEST_ROOT / variant / rid / 'compliance_report.json'
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding='utf-8'))

def _phase_discs(report, phase):
    return report['phase_results'][phase]['artifacts']['disclosure_verdicts']

def _classify_disc(disc):
    overall = (disc.get('overall') or '').lower()
    notes = (disc.get('notes') or '').lower()
    n_rv = len(disc.get('requirement_verdicts') or [])
    return {
        'is_omitted': overall == 'omitted' or 'omitted' in notes,
        'is_external_ref': overall == 'external_verification',
        'is_no_page_pack': 'no page pack' in notes,
        'is_empty_verdicts': n_rv == 0,
        'notes': disc.get('notes') or '',
    }

verdict_rows, disc_rows = [], []
for variant in VARIANTS:
    for rid in REPORT_IDS:
        report = _load_report(variant, rid)
        if report is None:
            print(f'[warn] thiếu {variant}/{rid}')
            continue
        for phase in _PAIR_PHASES:
            seen_v, seen_d = {}, {}
            for disc in _phase_discs(report, phase):
                did = disc['disclosure_id']
                mt = disc.get('material_topic')
                info = _classify_disc(disc)
                occ_d = seen_d.get((did, mt), 0)
                seen_d[(did, mt)] = occ_d + 1
                disc_rows.append({
                    'variant': variant, 'report_id': rid, 'phase': phase,
                    'disclosure_id': did, 'material_topic': mt, 'occurrence_idx': occ_d,
                    **info,
                })
                for rv in disc.get('requirement_verdicts') or []:
                    req_id = rv['requirement_id']
                    occ_v = seen_v.get((did, mt, req_id), 0)
                    seen_v[(did, mt, req_id)] = occ_v + 1
                    verdict_rows.append({
                        'variant': variant, 'report_id': rid, 'phase': phase,
                        'disclosure_id': did, 'material_topic': mt,
                        'requirement_id': req_id, 'occurrence_idx': occ_v,
                        'status': rv.get('status'),
                    })

verdicts = pd.DataFrame(verdict_rows)
disclosures = pd.DataFrame(disc_rows)
verdicts['material_topic'] = verdicts['material_topic'].fillna(MATERIAL_TOPIC_NA_SENTINEL)
disclosures['material_topic'] = disclosures['material_topic'].fillna(MATERIAL_TOPIC_NA_SENTINEL)

pivot = verdicts[_JOIN].drop_duplicates().reset_index(drop=True)
for v in VARIANTS:
    sub = verdicts.loc[verdicts['variant'] == v, _JOIN + ['status']].rename(columns={'status': f'{v}_status'})
    sub = sub.drop_duplicates(_JOIN, keep='last')
    pivot = pivot.merge(sub, on=_JOIN, how='left')

disc_lookup = disclosures.set_index(['variant'] + _DISC_KEYS)[
    ['is_empty_verdicts', 'is_omitted', 'is_external_ref', 'is_no_page_pack', 'notes']
].to_dict(orient='index')
g = disclosures.groupby(_DISC_KEYS)['is_empty_verdicts']
disc_consistency = {k: {'all_empty': bool(v)} for k, v in (g.sum() == g.count()).items()}

def _resolve(raw, variant, rid, did, mt, occ):
    if isinstance(raw, str):
        return raw
    key = (rid, did, mt, occ)
    info = disc_lookup.get((variant, *key))
    cons = disc_consistency.get(key)
    if info is None or cons is None:
        return '_missing_'
    if info['is_omitted']:
        return '_omitted_'
    if info['is_external_ref']:
        return '_external_ref_'
    if 'GCI row has missing' in info['notes']:
        return '_gci_error_'
    if info['is_no_page_pack']:
        return '_no_pack_consistent_' if cons['all_empty'] else _TAG_UNREC
    if info['is_empty_verdicts']:
        return _TAG_UNREC
    return '_missing_'

for v in VARIANTS:
    pivot[f'{v}_status_resolved'] = [
        _resolve(raw, v, rid, did, mt, int(occ))
        for raw, rid, did, mt, occ in zip(
            pivot[f'{v}_status'],
            pivot['report_id'], pivot['disclosure_id'],
            pivot['material_topic'], pivot['occurrence_idx'],
        )
    ]

resolved = pivot[[f'{v}_status_resolved' for v in VARIANTS]]
real = resolved.where(resolved.isin(_REAL))
n_disagreements = int((real.nunique(axis=1) > 1).sum())
print(f'disagreements: {n_disagreements:,}')

disagreements: 2,365
